[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_10/01_tag_10_gan_fashion_mnist.ipynb)

# Tag 10 – 01: Ein GAN trainieren: Fashion-MNIST erzeugen

Ein **Generative Adversarial Network (GAN)** besteht aus zwei Netzen mit gegensätzlichen Zielen: Der Generator wandelt Zufallsvektoren in Bilder um, der Diskriminator soll echte von erzeugten Bildern unterscheiden. Beide verbessern sich im Wechsel. Anders als bei der Klassifikation gibt es für ein neues Bild kein einzelnes Zielbild als Label.

## Lernziel und Formen

| Baustein | Input | Output | Ziel |
| --- | --- | --- | --- |
| Generator $G$ | Zufallsvektor $z$: `(100, 1, 1)` | künstliches Graustufenbild: `(1, 64, 64)` | Diskriminator täuschen |
| Diskriminator $D$ | echtes oder künstliches Bild | ein Logit | echt als 1, künstlich als 0 erkennen |

Fashion-MNIST enthält 28×28-Graustufenbilder von Kleidungsstücken. Wir skalieren sie auf 64×64 und normieren auf `[-1, 1]`, damit die `tanh`-Ausgabe des Generators exakt zur Datenverteilung passt.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_SIZE = 64
LATENT_DIM = 100
BATCH_SIZE = 128 if torch.cuda.is_available() else 32
EPOCHS = 8
# Für einen Kursdurchlauf bewusst begrenzt; None verwendet alle 60.000 Bilder.
MAX_TRAIN_IMAGES = 20_000

print('PyTorch:', torch.__version__)
print('Gerät:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # [0, 1] -> [-1, 1]
])

dataset = datasets.FashionMNIST(
    root='datasets', train=True, transform=transform, download=True
)
if MAX_TRAIN_IMAGES is not None:
    dataset = Subset(dataset, range(MAX_TRAIN_IMAGES))

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
images, labels = next(iter(loader))
print('Trainingsbilder:', len(dataset))
print('Batch-Form:', tuple(images.shape), 'Wertebereich:', (images.min().item(), images.max().item()))

grid = make_grid(images[:32], nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(10, 5))
plt.imshow(grid.permute(1, 2, 0), cmap='gray')
plt.title('Echte Trainingsbilder')
plt.axis('off');

## Architektur: Transposed Convolutions erzeugen Pixel

Der Generator vergrößert eine kleine räumliche Repräsentation schrittweise. Der Diskriminator macht die umgekehrte Richtung: Strided Convolutions verdichten das Bild zu einer Entscheidung. `BatchNorm` stabilisiert den Generator, `LeakyReLU` verhindert im Diskriminator völlig inaktive Neuronen.

Der Diskriminator gibt **keine Sigmoid-Wahrscheinlichkeit**, sondern einen Logit aus. `BCEWithLogitsLoss` kombiniert Sigmoid und Binary Cross Entropy numerisch stabil.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.network = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, 4, 1, 0, bias=False),  # 1 -> 4
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),         # 4 -> 8
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),          # 8 -> 16
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),           # 16 -> 32
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1, bias=False),            # 32 -> 64
            nn.Tanh(),
        )

    def forward(self, z):
        return self.network(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),  # 64 -> 32
            nn.Conv2d(32, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0),  # (N, 1, 4, 4) -> (N, 1, 1, 1)
        )

    def forward(self, image):
        return self.network(image).flatten()


generator = Generator().to(DEVICE)
discriminator = Discriminator().to(DEVICE)
count = lambda model: sum(parameter.numel() for parameter in model.parameters())
print(f'Generator-Parameter: {count(generator):,}')
print(f'Diskriminator-Parameter: {count(discriminator):,}')
assert generator(torch.randn(2, LATENT_DIM, 1, 1, device=DEVICE)).shape == (2, 1, 64, 64)

## Das Gegenspiel trainieren

Pro Batch passieren zwei getrennte Updates:

1. **Diskriminator:** echte Bilder bekommen Ziel `1`, vom Generator erzeugte Bilder Ziel `0`. Beim Update werden die künstlichen Bilder mit `.detach()` vom Generator getrennt.
2. **Generator:** neue Zufallsvektoren erzeugen Bilder. Jetzt ist das Ziel ebenfalls `1`: Die künstlichen Bilder sollen vom Diskriminator für echt gehalten werden.

Die Losses dürfen nicht isoliert interpretiert werden. Ein guter GAN-Trainingslauf hat kein klassisches, stetig gegen null fallendes Loss-Ziel. Entscheidend sind die erzeugten Bilder und ein ungefähr ausbalanciertes Gegenspiel.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_noise = torch.randn(32, LATENT_DIM, 1, 1, device=DEVICE)
history = {'generator': [], 'discriminator': []}

for epoch in range(EPOCHS):
    generator.train()
    discriminator.train()
    total_g = total_d = 0.0

    for real_images, _ in loader:
        real_images = real_images.to(DEVICE, non_blocking=True)
        batch_size = len(real_images)
        real_targets = torch.ones(batch_size, device=DEVICE)
        fake_targets = torch.zeros(batch_size, device=DEVICE)

        # 1. Diskriminator: echt -> 1, künstlich -> 0
        optimizer_d.zero_grad()
        real_loss = criterion(discriminator(real_images), real_targets)
        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
        fake_images = generator(noise)
        fake_loss = criterion(discriminator(fake_images.detach()), fake_targets)
        loss_d = real_loss + fake_loss
        loss_d.backward()
        optimizer_d.step()

        # 2. Generator: künstliche Bilder sollen als echt gelten
        optimizer_g.zero_grad()
        loss_g = criterion(discriminator(fake_images), real_targets)
        loss_g.backward()
        optimizer_g.step()

        total_d += loss_d.item()
        total_g += loss_g.item()

    history['discriminator'].append(total_d / len(loader))
    history['generator'].append(total_g / len(loader))
    print(f'Epoche {epoch + 1:02d}/{EPOCHS}: D={history["discriminator"][-1]:.3f}, G={history["generator"][-1]:.3f}')

    generator.eval()
    with torch.no_grad():
        preview = generator(fixed_noise).cpu()
    grid = make_grid(preview, nrow=8, normalize=True, value_range=(-1, 1))
    plt.figure(figsize=(10, 5))
    plt.imshow(grid.permute(1, 2, 0), cmap='gray')
    plt.title(f'Erzeugte Bilder nach Epoche {epoch + 1}')
    plt.axis('off')
    plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['discriminator'], marker='o', label='Diskriminator')
plt.plot(history['generator'], marker='o', label='Generator')
plt.xlabel('Epoche')
plt.ylabel('BCE Loss')
plt.title('GAN-Losses: ein Gegenspiel, kein klassischer Validierungs-Loss')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

generator.eval()
with torch.no_grad():
    new_images = generator(torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)).cpu()
grid = make_grid(new_images, nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0), cmap='gray')
plt.title('Neue, nur aus Zufall erzeugte Fashion-MNIST-Bilder')
plt.axis('off');

## Zusammenfassung

- Ein GAN lernt ohne Zielbild pro erzeugtem Bild: Der Diskriminator liefert das Trainingssignal.
- Der Generator mappt einen latenten Zufallsraum auf Bilder; ähnliche Zufallsvektoren können ähnliche Bildmuster erzeugen.
- GANs können scharfe Bilder erzeugen, sind aber beim Training empfindlich: Der Diskriminator darf nicht zu weit davonziehen, und **Mode Collapse** kann die Vielfalt reduzieren.
- Im nächsten Notebook wird statt eines Gegenspielers ein U-Net darauf trainiert, schrittweise Rauschen zu entfernen. Text-Embeddings steuern dabei, welches Bild entstehen soll.